# Pipeline RAG — Voice of Customer Intelligence

Este notebook demonstra o pipeline completo de **Retrieval-Augmented Generation (RAG)** sobre as avaliações do dataset Olist.

**Pré-requisitos:**
1. `01_exploracao.ipynb` executado → `data/processed/documents.json`
2. `02_base_conhecimento.ipynb` executado → `data/processed/reviews.index` e `documents.pkl`
3. `OPENAI_API_KEY` definida em um arquivo `.env` na raiz do projeto

**Fluxo do pipeline:**
```
Pergunta do usuário
       ↓
Embedding da query (sentence-transformers)
       ↓
Busca semântica no índice FAISS
       ↓
Filtro por limiar de similaridade (≥ 0.30)
       ↓
Construção do contexto com as avaliações recuperadas
       ↓
Geração de resposta pelo LLM (gpt-4o-mini)
       ↓
Resposta fundamentada + evidências rastreáveis
```

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

sys.path.insert(0, str(Path().resolve().parent))

load_dotenv(Path("../.env"))  # carrega OPENAI_API_KEY do arquivo .env

from src.retrieval import ReviewSearcher
from src.rag import answer, build_context

DATA_PROCESSED = Path("../data/processed")

assert (DATA_PROCESSED / "reviews.index").exists(), (
    "Índice não encontrado. Execute 02_base_conhecimento.ipynb primeiro."
)

searcher = ReviewSearcher(DATA_PROCESSED)
client = OpenAI()

print(f"Índice carregado: {searcher.index.ntotal:,} avaliações indexadas")

## 1. Recuperação semântica

Antes de gerar qualquer resposta, vamos entender o que acontece na etapa de recuperação: a pergunta é convertida em um vetor e o FAISS retorna as avaliações mais próximas no espaço semântico.

In [ ]:
query = "problemas com atraso na entrega"

results = searcher.search(query, k=5)

print(f"Query: '{query}'")
print(f"Avaliações recuperadas: {len(results)}")
print()
for i, r in enumerate(results, 1):
    meta = r.get('metadata', {})
    print(f"  [{i}] Nota: {meta.get('review_score', '?')}/5  |  Similaridade: {r['similarity_score']:.3f}")
    text = r['text']
    print(f"      {text[:200]}..." if len(text) > 200 else f"      {text}")
    print()

## 2. Construção do contexto

As avaliações recuperadas são formatadas em um bloco de contexto estruturado antes de serem enviadas ao LLM. Cada avaliação é numerada e acompanhada de sua nota e similaridade, permitindo rastreabilidade.

In [ ]:
context = build_context(results)
print(context[:1000], "..." if len(context) > 1000 else "")

## 3. Pipeline RAG completo

A função `answer()` encapsula todo o fluxo: recuperação → filtragem → contexto → LLM.

A função auxiliar `demo_answer()` abaixo imprime a resposta e as evidências de forma legível.

In [ ]:
def demo_answer(question: str, k: int = 15) -> None:
    """Executa o pipeline RAG e imprime resposta + evidências."""
    print(f"{'='*70}")
    print(f"PERGUNTA: {question}")
    print('='*70)

    result = answer(question, searcher, k=k, client=client)

    print("\nRESPOSTA:")
    print(result["answer"])

    print(f"\nEVIDÊNCIAS UTILIZADAS ({len(result['evidence'])} avaliações recuperadas):")
    if not result["has_evidence"]:
        print("  (nenhuma evidência suficiente encontrada)")
    else:
        for i, doc in enumerate(result["evidence"], 1):
            meta = doc.get('metadata', {})
            print(f"  [{i}] Nota: {meta.get('review_score', '?')}/5  |  Similaridade: {doc['similarity_score']:.3f}")
            text = doc['text']
            print(f"      {text[:180]}..." if len(text) > 180 else f"      {text}")
    print()

## 4. Exemplos de perguntas

### 4.1 Investigação de reclamações

In [ ]:
demo_answer("Quais são os principais problemas relatados pelos clientes em suas avaliações?")

### 4.2 Tema específico — entrega

In [ ]:
demo_answer("O que os clientes relatam sobre problemas relacionados à entrega?")

### 4.3 Busca por experiências positivas

In [ ]:
demo_answer("Quais aspectos da experiência de compra são mais elogiados pelos clientes?")

### 4.4 Investigação de insatisfação

In [ ]:
demo_answer("Quais padrões podem ser identificados nas avaliações de clientes insatisfeitos?")

### 4.5 Exploração baseada em evidências

In [ ]:
demo_answer("Quais avaliações sustentam a conclusão de que existem problemas recorrentes relacionados ao prazo de entrega?")

## 5. Tratamento de perguntas sem evidências suficientes

Quando a pergunta está fora do escopo da base de conhecimento — ou quando não existem avaliações suficientemente similares — o sistema reconhece suas limitações em vez de inventar uma resposta.

In [ ]:
# Pergunta fora do escopo da base (não há avaliações sobre clima ou política)
demo_answer("Qual foi o impacto da taxa de câmbio no preço dos produtos em 2017?")

In [ ]:
# Pergunta muito específica que pode não ter evidências suficientes
demo_answer("O que os clientes pensam sobre o serviço de montagem de móveis?")

## 6. Exploração com filtros de metadados (desafio adicional)

Utilizamos os metadados enriquecidos para perguntas mais contextualizadas, filtrando as avaliações antes da busca ou incluindo informações de contexto na análise.

In [ ]:
# Pergunta específica sobre avaliações negativas (nota 1)
demo_answer("Quais são os principais problemas relatados nas avaliações com nota 1 ou 2?")

In [ ]:
# Exploração por categoria de produto
demo_answer("O que os clientes relatam sobre a qualidade dos produtos de eletrônicos?")